In [ ]:
import cv2
from matplotlib import pyplot as plt
import numpy as np
import torch
from torch.nn import functional as F
import kornia.geometry.subpix.dsnt as dsnt
from kornia.utils.grid import create_meshgrid
import math
from einops.einops import rearrange
from typing import Sequence

from utils.misc import setup_gpus
from default_config import get_cfg_defaults
from maff.maff import MAFF
from datasets.overall_dataset import MAFF_Dataset

In [ ]:
print(f"{torch.tensor([1.0]).item(): .6f}")

In [ ]:
config = get_cfg_defaults()
config.DEVICE.GPU_IDX = "2,"
config.DEVICE.ENABLE_DDP = False
# setup exact gpus available and set CUDA_VISIBLE_DEVICES variable
n_gpu_available = (
    setup_gpus(config.DEVICE.GPU_IDX) if config.DEVICE.ENABLE_GPU else 0
)
config.TRAINER.WORLD_SIZE = n_gpu_available * config.DEVICE.NUM_NODES
config.TRAINER.TRUE_BATCH_SIZE = (
    config.TRAINER.WORLD_SIZE * config.LOADER.BATCH_SIZE
)
config.TRAINER.SCALING = (
    config.TRAINER.TRUE_BATCH_SIZE / config.TRAINER.CANONICAL_BS
)
config.TRAINER.TRUE_LR = config.TRAINER.CANONICAL_LR * config.TRAINER.SCALING
config.TRAINER.WARMUP_STEP = math.floor(
    config.TRAINER.WARMUP_STEP / config.TRAINER.SCALING
)

In [ ]:
device = torch.device("cuda:0")
maff = MAFF(config=config["MODEL"])
maff.to(device)

In [ ]:
data_module = MAFF_Dataset(config=config)
data_module.setup(stage="fit")
train_dataloader = data_module.train_dataloader()
val_dataloader = data_module.val_dataloader()

In [ ]:
for data in train_dataloader:
    break

In [ ]:
data.update(
    {
        "image0": data["image0"].to(device),
        "image1": data["image1"].to(device),
        "mask0": data["mask0"].to(device),
        "mask1": data["mask1"].to(device)
    }
)

In [ ]:
torch.cuda.empty_cache()

In [ ]:
maff(data)

In [ ]:
sim_matrix = data["sim_matrix"]
print(f"GPU memory usage: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

In [ ]:
conf_matrix = F.softmax(sim_matrix, 1) * F.softmax(sim_matrix, 2)
data.update({"conf_matrix": conf_matrix})
print(f"GPU memory usage: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

In [ ]:
conf_matrix = conf_matrix + 1e-9
print(f"GPU memory usage: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

In [ ]:
torch.cuda.empty_cache()
print(f"GPU memory usage: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

In [ ]:
feat0 = data["feat0_c"]
feat1 = data["feat1_c"]
b_idx_c = data["b_idx_c"]
i_idx_c = data["i_idx_c"]
j_idx_c = data["j_idx_c"]
scale = data["hw0_i"][0] / data["hw0_c"][0]
H, W = data["hw0_c"]
C = feat0.shape[-1]
window_size = maff.config.FINE_MATCHING.WINDOW_SIZE  # window size

In [ ]:
# 1. Pick feature from coarse feature map x0 using coarse match
feat0_picked = feat0[b_idx_c, i_idx_c]  # M, C

In [ ]:
_feat=feat1
H, W =data["hw1_c"]
b_idx=b_idx_c
j_idx=j_idx_c
pad_size = window_size // 2

# Rearrange
feat = rearrange(_feat, "b (h w) c -> b c h w", h=H, w=W)

# Padding
feat_padded = F.pad(feat, (pad_size, pad_size, pad_size, pad_size))

In [ ]:
# Calculate row and column indices
row_indices = j_idx // W
col_indices = j_idx % W

# Create grid indices
row_offsets = torch.arange(window_size, device=feat.device)
col_offsets = torch.arange(window_size, device=feat.device)

row_indices = row_indices.unsqueeze(1) + row_offsets.unsqueeze(0)
col_indices = col_indices.unsqueeze(1) + col_offsets.unsqueeze(0)

In [ ]:
windows = feat_padded[
    b_idx[:, None, None, None],
    :,
    row_indices[:, None, :, None],
    col_indices[:, None, None, :],
]

In [ ]:
# Old method
windows = []
for i in range(b_idx.shape[0]):
    window = feat_padded[
        b_idx[i],
        :,
        j_idx[i] // W : j_idx[i] // W + window_size,
        j_idx[i] % W : j_idx[i] % W + window_size,
    ]
    windows.append(window)

In [ ]:
windows = torch.stack(windows)


In [ ]:
coarse_coord_0 = data["coarse_coord_0"]
coarse_coord_1 = data["coarse_coord_1"]
fine_coord_0 = data["fine_coord_0"]
fine_coord_1 = data["fine_coord_1"]

In [ ]:
# 从 data 中获取图像并转换为 numpy 数组
image0 = data["image0"].detach().cpu().numpy()[0, 0] * 255
W1 = data["hw0_i"][1]
image1 = data["image1"].detach().cpu().numpy()[0, 0] * 255

# 确保图像数据类型为 uint8
image0 = image0.astype(np.uint8)
image1 = image1.astype(np.uint8)

# 将灰度图像转换为 BGR 图像
image0 = cv2.cvtColor(image0, cv2.COLOR_GRAY2BGR)
image1 = cv2.cvtColor(image1, cv2.COLOR_GRAY2BGR)

# 将图像插值至两倍大小
image0 = cv2.resize(image0, (image0.shape[1] * 2, image0.shape[0] * 2), interpolation=cv2.INTER_LINEAR)
image1 = cv2.resize(image1, (image1.shape[1] * 2, image1.shape[0] * 2), interpolation=cv2.INTER_LINEAR)
W1 *= 2

# 将两张图像水平拼接在一起
combined_image = cv2.hconcat([image0, image1])

# 预定义一组常规的颜色
colors = [
    (255, 0, 0),    # 红色
    (0, 255, 0),    # 绿色
    (0, 0, 255),    # 蓝色
    (255, 255, 0),  # 黄色
    (255, 0, 255),  # 洋红
    (0, 255, 255),  # 青色
    (128, 0, 0),    # 深红
    (0, 128, 0),    # 深绿
    (0, 0, 128),    # 深蓝
    (128, 128, 0),  # 橄榄绿
    (128, 0, 128),  # 紫色
    (0, 128, 128),  # 青绿色
    (192, 192, 192),# 银色
    (128, 128, 128),# 灰色
    (0, 0, 0),      # 黑色
    (255, 165, 0),  # 橙色
    (255, 20, 147), # 深粉色
    (75, 0, 130),   # 靛蓝色
    (240, 230, 140),# 卡其色
    (173, 216, 230) # 淡蓝色
]

# 在图像上绘制圆圈和连线
for idx, ((x0, y0), (x1, y1)) in enumerate(zip(fine_coord_0, fine_coord_1)):
    # 从预定义的颜色中随机选择一个颜色
    color = colors[np.random.randint(len(colors))]
    
    # 调整坐标至两倍大小
    x0, y0, x1, y1 = x0 * 2, y0 * 2, x1 * 2, y1 * 2
    
    # 绘制圆圈
    cv2.circle(combined_image, (int(x0), int(y0)), 2, color, 1, lineType=cv2.LINE_AA)
    cv2.circle(combined_image, (int(x1) + W1, int(y1)), 2, color, 1, lineType=cv2.LINE_AA)
    
    # 每50个点绘制一次连线
    if idx % 50 == 0:
        cv2.line(combined_image, (int(x0), int(y0)), (int(x1) + W1, int(y1)), (0, 255, 0), 1, lineType=cv2.LINE_AA)

# 保存图像到文件
output_path = 'sample_match.png'
cv2.imwrite(output_path, combined_image)

# 使用 matplotlib 显示拼接后的图像
plt.imshow(cv2.cvtColor(combined_image, cv2.COLOR_BGR2RGB))
plt.axis('off')  # 关闭坐标轴
plt.show()

In [ ]:
x0 = data["feat0_c"]
x1 = data["feat1_c"]
sim_matrix = data["sim_matrix"]

scale = data["hw0_i"][0] / data["hw0_c"][0]

In [ ]:
with torch.no_grad():
    # Get coarse matches
    coarse_match = torch.argwhere(
        sim_matrix / sim_matrix.max() > 0.8
    )  # M, 3(B, I, J)

    b_idx_c = coarse_match[:, 0]
    i_idx_c = coarse_match[:, 1]
    j_idx_c = coarse_match[:, 2]
    torch.cuda.empty_cache()

    # Match indices -> coordinates
    scale = data["hw0_i"][0] / data["hw0_c"][0]
    coarse_coord_0 = (
        torch.stack(
            (
                i_idx_c % data["hw0_c"][0],
                i_idx_c // data["hw0_c"][0],
            ),
            dim=1,
        )
        * scale
    )
    coarse_coord_1 = (
        torch.stack(
            (
                j_idx_c % data["hw1_c"][0],
                j_idx_c // data["hw1_c"][0],
            ),
            dim=1,
        )
        * scale
    )

    data.update(
        {
            "b_idx_c": b_idx_c,  # in coarse coordinate
            "i_idx_c": i_idx_c,  # in coarse coordinate
            "j_idx_c": j_idx_c,  # in coarse coordinate
            "coarse_coord_0": coarse_coord_0,  # in absolute coordinate
            "coarse_coord_1": coarse_coord_1,  # in absolute coordinate
        }
    )

In [ ]:
def extract_feat_window(
    _feat: torch.Tensor,
    feat_hw: Sequence[int],
    b_idx: torch.Tensor,
    j_idx: torch.Tensor,
    window_size: int,
):
    H, W = feat_hw
    pad_size = window_size // 2
    # Rearrange
    feat = rearrange(_feat, "b (h w) c -> b c h w", h=H, w=W)

    # Padding
    feat_padded = F.pad(feat, (pad_size, pad_size, pad_size, pad_size))

    windows = []
    for i in range(b_idx.shape[0]):
        window = feat_padded[
            b_idx[i],
            :,
            j_idx[i] // W : j_idx[i] // W + window_size,
            j_idx[i] % W : j_idx[i] % W + window_size,
        ]
        windows.append(window)

    return torch.stack(windows)


H, W = data["hw0_c"]
B = x0.shape[0]
M = b_idx_c.shape[0]
C = x0.shape[-1]
scale = data["hw0_i"][0] / data["hw0_c"][0]
window_size = config.MODEL.FINE_MATCHING.WINDOW_SIZE  # window size

# Pick feature from coarse feature map x0
feat0_picked = x0[b_idx_c, i_idx_c]  # M, C
# Pick a window of feature from coarse feature map x1
feat1_window_picked = extract_feat_window(
    _feat=x1,
    feat_hw=data["hw1_c"],
    b_idx=b_idx_c,
    j_idx=j_idx_c,
    window_size=window_size,
)  # M, C, window_size, window_size

# Correlation between both feature
window_heatmap = torch.einsum("mc,mchw->mhw", feat0_picked, feat1_window_picked)
window_heatmap = torch.softmax(window_heatmap / (C**0.5), dim=1)

# Compute coordinates from heatmap
coords_normalized = dsnt.spatial_expectation2d(window_heatmap[None], True)[0]  # M, 2
grid_normalized = create_meshgrid(
    window_size, window_size, True, window_heatmap.device
).reshape(1, -1, 2)  # [1, WW, 2]

# Compute absolute coordinates
fine_coord_0 = data["coarse_coord_0"]
fine_coord_1 = (
    torch.concat(
        (
            torch.zeros(M, 1, device=coords_normalized.device),
            (coords_normalized * (window_size // 2) * scale),
        ),
        dim=1,
    )
    + data["coarse_coord_1"]
)

data.update({"fine_coord_0": fine_coord_0, "fine_coord_1": fine_coord_1})

# compute std over <x, y>
var = (
    torch.sum(grid_normalized**2 * window_heatmap.view(-1, window_size**2, 1), dim=1)
    - coords_normalized**2
)  # [M, 2]
std = torch.sum(
    torch.sqrt(torch.clamp(var, min=1e-10)), -1
)  # [M]  clamp needed for numerical stability


In [ ]:
H, W = data["hw0_c"]
window_size = config.MODEL.FINE_MATCHING.WINDOW_SIZE  # window size

# Pick a window of similarity matrix using coarse coordinates
sim_matrix_window = rearrange(
    sim_matrix[b_idx_c, i_idx_c], "m (H1 W1) -> m H1 W1", H1=H, W1=W
)
sim_matrix_window = F.unfold(
    sim_matrix_window,
    kernel_size=(window_size, window_size),
    stride=(1, 1),
    padding=window_size // 2,
)
sim_matrix_window = rearrange(
    sim_matrix_window, "(m ww) hw -> m ww hw", ww=window_size**2
)
j_idx_expanded = j_idx_c.view(-1, 1, 1).repeat(1, window_size**2, 1)
sim_matrix_window = torch.gather(
    sim_matrix_window,
    index=j_idx_expanded,
    dim=2,
).squeeze(2)
torch.cuda.empty_cache()

In [ ]:
import torch
import kornia.geometry.subpix.dsnt as dsnt

# 创建一个示例热图
heatmap = torch.softmax(
    torch.tensor([[0., 0., 0.], [0., 1., 0.], [0., 0., 0.]]), dim=1
).view(1, 1, 3, 3)

# 使用 DSNT 提取坐标
coords = dsnt.spatial_expectation2d(heatmap, True) * (W//2)

print(coords)

In [ ]:
compute_supervision_coarse(data, 4)

image0 = data["image0"].detach().cpu().numpy()[0, 0]
image1 = data["image1"].detach().cpu().numpy()[0, 0]
conf_matrix_gt = data["conf_matrix_gt"]

_, pos_x, pos_y = torch.where(conf_matrix_gt==1.0)

pos_image0_x = pos_x % 160
pos_image0_y = pos_x // 160
pos_image1_x = pos_y % 160
pos_image1_y = pos_y // 160

image0 = cv2.cvtColor(image0, cv2.COLOR_GRAY2BGR)
image1 = cv2.cvtColor(image1, cv2.COLOR_GRAY2BGR)

for x, y in zip(pos_image0_x, pos_image0_y):
    cv2.circle(image0, (int(x) * 4, int(y) * 4), 4, (0,0,255), 2)
    
for x, y in zip(pos_image1_x, pos_image1_y):
    cv2.circle(image1, (int(x) * 4, int(y) * 4), 4, (0,0,255), 2)

In [ ]:
input0 = torch.randint(low = 0, high = 256, size=(1, 1, config["IMAGE_SIZE"], config["IMAGE_SIZE"]), dtype=torch.float32).to(device)
input1 = torch.randint(low = 0, high = 256, size=(1, 1, config["IMAGE_SIZE"], config["IMAGE_SIZE"]), dtype=torch.float32).to(device)
mask0 = torch.randint(low = 0, high = 10, size=(1, int(config["IMAGE_SIZE"] / 4), int(config["IMAGE_SIZE"] / 4)), dtype=torch.float32).to(device)
mask1 = torch.randint(low = 0, high = 10, size=(1, int(config["IMAGE_SIZE"] / 4), int(config["IMAGE_SIZE"] / 4)), dtype=torch.float32).to(device)
data = {
    "image0": input0,
    "image1": input1,
    "mask0" : mask0,
    "mask1": mask1
}

In [ ]:
conf_matrix = maff(data)

In [ ]:
sim_matrix = torch.einsum("blc,bsc->bls", x1, x2) / 0.1

In [ ]:
conf_matrix = F.softmax(sim_matrix, 1) * F.softmax(sim_matrix, 2)

In [ ]:
print(f":", torch.cuda.memory_allocated(torch.device("cuda:3"))/1024/1024/1024)

In [ ]:
torch.cuda.empty_cache()

In [ ]:
from einops.einops import rearrange
for i, scale in enumerate(x1):
    # [B x C x H x W] -> [B x (HW) x C]
    x1[i]= rearrange(scale, 'b c h w -> b (h w) c')

In [ ]:
x1 = torch.concat(x1, dim=1)

In [ ]:
x1, x2 = pe(x1, x2)

In [ ]:
d_models = [64, 128, 256, 512]
max_hw = 840
scales = [0.5, 0.25, 0.125, 0.0625]
dtype = torch.float32
num_scales = len(scales)

In [ ]:
pos_x = torch.arange(max_hw, dtype=dtype)
pos_y = torch.arange(max_hw, dtype=dtype)
pos_z = torch.arange(num_scales, dtype=dtype)

In [ ]:
all_scales_pos_emb = []

In [ ]:
pos_z[0]

In [ ]:
for i in range(num_scales):
    d_pos_embeddings = int(np.ceil(d_models[i] / 3))
    inv_freq = 1.0 / (10000.0 ** (torch.arange(0, d_pos_embeddings, 2, dtype=dtype) / d_pos_embeddings))
    hw_pos_emb = torch.zeros(size=(max_hw, max_hw, 6 * inv_freq.shape[0]), dtype=dtype)                      # H, W, C
    
    temp_x = torch.einsum("i,j->ij", pos_x, inv_freq)
    temp_y = torch.einsum("i,j->ij", pos_y, inv_freq)
    temp_z = pos_z[i] * inv_freq
    
    sin_x = torch.sin(temp_x).unsqueeze(0)
    sin_y = torch.sin(temp_y).unsqueeze(1)
    sin_z = torch.sin(temp_z).unsqueeze(0).unsqueeze(0)
    cos_x = torch.cos(temp_x).unsqueeze(0)
    cos_y = torch.cos(temp_y).unsqueeze(1)
    cos_z = torch.cos(temp_z).unsqueeze(0).unsqueeze(0)
    
    hw_pos_emb[:, :, 0::6] = sin_x
    hw_pos_emb[:, :, 1::6] = cos_x
    hw_pos_emb[:, :, 2::6] = sin_y
    hw_pos_emb[:, :, 3::6] = cos_y
    hw_pos_emb[:, :, 4::6] = sin_z
    hw_pos_emb[:, :, 5::6] = cos_z

    # Crop the exceeding channels
    hw_pos_emb = hw_pos_emb[:, :, 0: d_models[i]]
    
    hw_pos_emb_rescaled = torch.nn.functional.interpolate(hw_pos_emb.swapaxes(0, -1).unsqueeze(0), scale_factor=(scales[i], scales[i])).squeeze(0).swapaxes(0, -1)
    
    all_scales_pos_emb.append(hw_pos_emb_rescaled)

In [ ]:
[print(i.shape) for i in all_scales_pos_emb]
temp_0 = all_scales_pos_emb[0]
temp_1 = all_scales_pos_emb[1]
temp_2 = all_scales_pos_emb[2]
temp_3 = all_scales_pos_emb[3]

In [ ]:
temp = torch.stack(all_scales_pos_emb).unsqueeze

In [ ]:
temp=torch.sin(temp_x)

In [ ]:
channels = 12
channels = int(np.ceil(channels / 6) * 2)
print(channels)

In [ ]:
inv_freq = 1.0 / (10000 ** (torch.arange(start=0, end=channels, step=2, dtype=torch.float32) / channels))
print(inv_freq)

In [ ]:
y_position = torch.ones(128).cumsum(0).float().unsqueeze(0)
print(y_position)

In [ ]:
x1 = torch.zeros(size=(3, 12, 512))
x2 = torch.zeros(size=(3, 15, 512))

for i in range(x1.shape[1]):
    x1[:, i, 0] = i
    
for i in range(x2.shape[1]):
    x2[:, i, 0] = i * 10

In [ ]:
concat_x = torch.concat(tensors=(x1, x2), dim=-2)
concat_x_flip = torch.flip(concat_x, dims=[-2])

fuse_x = torch.concat(tensors=(concat_x, concat_x_flip), dim=-2)

In [ ]:
fuse_y = fuse_x.clone()

In [ ]:
L = fuse_y.shape[1]
concat_y = 0.5*(fuse_y[:, 0:L//2, :] + fuse_y[:, L//2:, :].flip(dims=[1]))


In [ ]:
L1 = x1.shape[-2]
L2 = x2.shape[-2]
y1 = concat_y[:, 0:L1, :]
y2 = concat_y[:, L1:, :]

In [ ]:
class MambaEncoder(nn.Module):
    def __init__(
        self,
        layers_dim: Sequence[int],
        layers_type: Sequence[str],
        inner_expansion: float = 2,
        conv_dim: int = 4,
        device: torch.device = None,
        dtype: torch.dtype = None,
        using_mamba2: bool = False
    ):
        super(MambaEncoder, self).__init__()
        
        self.layers_dim: Sequence[int] = layers_dim
        self.layers_type: Sequence[str] = layers_type
        self.inner_expansion: float = inner_expansion
        self.conv_dim: int = conv_dim
        self.device: torch.device = device
        self.dtype: torch.dtype = dtype
        self.using_mamba2: bool = using_mamba2
        
        
        
        

In [ ]:
device = torch.device("cuda:4")
test_mamba = MambaLayer(device=device)

In [ ]:
print(test_mamba)

x = torch.zeros(size=(32, 1024, 1024)).to(device)

In [ ]:
y = test_mamba(x)

In [ ]:
print(y.shape)